In [1]:
# Import the libraries
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Load weekly Brent and the weekly ENSE reference prices
brent = pd.read_csv("../data/processed/brent_eur_weekly.csv", parse_dates=["date"])
ense = pd.read_csv("../data/processed/ense_retail_margins.csv", parse_dates=["date"])

# Move Brent dates from Sunday to Monday so both series refer to the same week
brent["date"] = brent["date"] + pd.Timedelta(days=1)
df = pd.merge(brent[["date", "brent_eur_litre"]], ense[["date", "gasoline_ref", "diesel_ref"]], on="date")
print(len(df), "weeks")

395 weeks


In [2]:
# Asymmetry test from Brent to the ENSE reference price (refining and wholesale stage)
results = []

for fuel_name in ["gasoline", "diesel"]:
    # Weekly change in Brent, split into rises and falls
    d_brent = df["brent_eur_litre"].diff()
    df["up"] = d_brent.clip(lower=0)
    df["down"] = d_brent.clip(upper=0)

    # The same changes in the previous three weeks
    for lag in [1, 2, 3]:
        df[f"up_lag{lag}"] = df["up"].shift(lag)
        df[f"down_lag{lag}"] = df["down"].shift(lag)

    # Weekly change in the reference price and last week's change
    df["d_ref"] = df[f"{fuel_name}_ref"].diff()
    df["d_ref_lag1"] = df["d_ref"].shift(1)

    # Regression with HAC standard errors
    formula = "d_ref ~ up + up_lag1 + up_lag2 + up_lag3 + down + down_lag1 + down_lag2 + down_lag3 + d_ref_lag1"
    model = smf.ols(formula, data=df).fit(cov_type="HAC", cov_kwds={"maxlags": 4})

    # Long run pass-through of rises and falls
    rho = model.params["d_ref_lag1"]
    sum_up = model.params["up"] + model.params["up_lag1"] + model.params["up_lag2"] + model.params["up_lag3"]
    sum_down = model.params["down"] + model.params["down_lag1"] + model.params["down_lag2"] + model.params["down_lag3"]
    lr_up = sum_up / (1 - rho)
    lr_down = sum_down / (1 - rho)

    # Symmetry test: are rises and falls passed on equally?
    test = model.f_test("up + up_lag1 + up_lag2 + up_lag3 = down + down_lag1 + down_lag2 + down_lag3")

    results.append({
        "fuel": fuel_name,
        "lr_up": round(lr_up, 2),
        "lr_down": round(lr_down, 2),
        "p_value": round(float(test.pvalue), 3),
        "r2": round(model.rsquared, 2),
        "n": int(model.nobs),
    })

# Save and show the results
results = pd.DataFrame(results)
results.to_csv("../data/processed/rockets_feathers_brent_to_ense.csv", index=False)
results

,fuel,lr_up,lr_down,p_value,r2,n
0,gasoline,0.99,1.10,0.722,0.47,391
1,diesel,1.48,1.06,0.179,0.53,391
